# Part 3 — Agentic RAG with LangGraph: CRAG State Machine

**Series:** Agentic RAG with LangGraph — ArXiv ML/AI Research Paper Q&A  
**Notebook:** 3 of 3  
**Prerequisite:** Run notebooks 01 and 02 first — this notebook loads the saved FAISS index.

---

## What you will build

By the end of this notebook you will have:

1. Understood what makes RAG *agentic* (adaptive decision-making vs. fixed pipelines)
2. Learned what LangGraph is and how to model a workflow as a **state machine**
3. Implemented the **CRAG (Corrective RAG)** architecture node by node
4. Built a graph with **7 nodes** and **conditional edges** that route between them
5. Run the agent on queries and inspected execution traces to see which paths fired
6. Produced the final **3-way comparison table**: Naive vs. Advanced vs. Agentic RAG

---

## Prerequisites — what you need to know before starting

| Concept | Where introduced |
|---------|----------------|
| Dense retrieval + FAISS | Notebook 01 |
| Hybrid retrieval + reranking | Notebook 02 |
| What a Python dataclass is | Standard Python |
| What JSON is | Standard |

**New concepts in this notebook — all explained from scratch:**
- State machines (nodes, edges, state)
- LangGraph (StateGraph, TypedDict, conditional_edges)
- CRAG architecture (Corrective RAG, 2024)
- LLM-as-judge (grading relevance and faithfulness)
- Web search as a tool / fallback

---

## Lesson 1 — What is agentic RAG?

### The pipeline vs. agent distinction

**Notebooks 01 and 02** built *pipelines*: a fixed sequence of steps that runs the same way for every query:

```
Pipeline RAG (fixed):
Query → Retrieve → Rerank → Generate → Answer
         (always k=5)  (always rerank)  (no quality check)
```

This has real problems:
- A query whose answer is NOT in the corpus still gets irrelevant passages injected into the prompt
- The LLM generates an answer even when the context is garbage
- There is no fallback when retrieval fails
- Every query — simple or complex — gets the same treatment

**Agentic RAG** replaces the fixed pipeline with a decision-making loop:

```
Agentic RAG (adaptive):
Query → Retrieve → Are docs relevant? ──yes──► Generate → Is answer faithful? ──yes──► Return
                        │                                         │
                        no                                        no
                        │                                         │
                        ▼                                         ▼
                  Web search                               Regenerate
```

The system **grades its own output at each step** and decides what to do next. This is what makes it "agentic."

### The CRAG paper

**Corrective Retrieval Augmented Generation (CRAG)** — Yan et al., 2024 — formalises this pattern:

1. Retrieve documents
2. Grade each document: **Correct** / **Ambiguous** / **Incorrect**
3. If Correct → generate from documents
4. If Ambiguous or Incorrect → supplement or replace with web search
5. Grade the generated answer for faithfulness
6. If hallucination detected → regenerate

CRAG showed consistent improvement over naive RAG across 4 knowledge-intensive QA benchmarks without any additional training.

---

## Lesson 2 — What is LangGraph?

### State machines — the foundation

A **state machine** is a model of computation with:
- A **state**: a data structure that holds all information about "where we are" and "what we know"
- **Nodes**: functions that read the state, do something, and return an updated state
- **Edges**: connections between nodes that define which node runs next
- **Conditional edges**: edges where the *destination* depends on the current state value

```
State = { question, documents, generation, hallucination_flag }

Node: retrieve
  Input state  : { question: "What is RAG?" }
  Action       : search FAISS index
  Output state : { question: "...", documents: [doc1, doc2, doc3] }

Conditional edge after grade_documents:
  if state["all_docs_relevant"] == True  → go to generate_answer
  if state["all_docs_relevant"] == False → go to web_search
```

### LangGraph

LangGraph is a Python framework that implements state machines for LLM workflows:

```python
from langgraph.graph import StateGraph

graph = StateGraph(MyState)          # define the state schema
graph.add_node("retrieve", retrieve_fn)         # add nodes
graph.add_node("grade", grade_fn)
graph.add_edge("retrieve", "grade")             # fixed edge
graph.add_conditional_edges(                    # conditional edge
    "grade",
    decide_fn,         # returns next node name
    {"generate": "generate", "search": "web_search"}
)
app = graph.compile()                # compile to a runnable
result = app.invoke({"question": "..."})  # run it
```

Key benefit over plain Python: LangGraph handles the execution loop, error recovery, streaming, and produces a full execution trace you can inspect — which node ran, what state it received, what state it returned.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
INDEX_DIR = ARTIFACTS_DIR / "faiss_index"
EVAL_DIR = ARTIFACTS_DIR / "eval_results"
TRACE_DIR = ARTIFACTS_DIR / "agent_traces"
TRACE_DIR.mkdir(exist_ok=True)

import json
import time
from typing import Literal
from typing_extensions import TypedDict

import ollama
from loguru import logger
from langgraph.graph import StateGraph, END
from duckduckgo_search import DDGS

from src.ingest import (
    load_index_and_chunks, load_arxiv_papers,
    EMBED_MODEL_PRIMARY, EMBED_MODEL_LITE,
)
from src.retriever import DenseRetriever, BM25Retriever, HybridRetriever, Reranker
from src.evaluator import EvalResults

# Load the index built in notebook 01
faiss_index, chunks = load_index_and_chunks(INDEX_DIR)

# CRITICAL: use the same embed_model that was used to build the index in notebook 01
EMBED_MODEL = EMBED_MODEL_LITE   # qwen3-embedding:0.6b — matches the saved index
LLM_MODEL   = "granite4.1:8b"

# Instantiate retrievers
dense_retriever  = DenseRetriever(faiss_index, chunks, embed_model=EMBED_MODEL)
bm25_retriever   = BM25Retriever(chunks)
hybrid_retriever = HybridRetriever(dense_retriever, bm25_retriever, alpha=0.7)
reranker         = Reranker()

print(f"Index loaded    : {faiss_index.ntotal} vectors, dim={faiss_index.d}")
print(f"Embed model     : {EMBED_MODEL}")
print(f"LLM model       : {LLM_MODEL}")
print("All components loaded.")

---

## Step 1 — Define the graph state

The **state** is the single data structure that flows through the entire graph. Every node reads from it and writes back to it. Think of it as the "shared memory" of the agent.

We use a `TypedDict` so every field is explicitly typed — this prevents bugs where one node produces a field with the wrong type that silently breaks a downstream node.

In [ ]:
# GraphState is the shared data structure that passes between nodes.
# Every field starts as None/empty and gets filled as the graph executes.

class GraphState(TypedDict):
    # The original user question — set at graph entry, never changed
    question: str

    # Documents retrieved from FAISS (list of chunk dicts from src/ingest.py)
    documents: list

    # After grading, only the "relevant" documents remain here
    filtered_documents: list

    # "relevant" | "irrelevant" | "ambiguous" — set by grade_documents node
    retrieval_grade: str

    # The final generated answer string
    generation: str

    # "faithful" | "hallucinated" — set by grade_hallucination node
    faithfulness_grade: str

    # How many times we've attempted generation (guards against infinite loops)
    generation_attempts: int

    # Log of which nodes ran and what decisions were made (for the trace viewer)
    execution_trace: list


print("State schema defined with fields:")
for field_name, field_type in GraphState.__annotations__.items():
    print(f"  {field_name:25s}: {field_type}")

---

## Step 2 — Build each node

A **node** is just a Python function:
- Input: the current `GraphState` dict
- Output: a dict of *only the fields it changed* (LangGraph merges this into the state)

We build 6 nodes:

| Node | What it does |
|------|-------------|
| `retrieve` | FAISS + BM25 hybrid search, return top-10 |
| `grade_documents` | LLM judges relevance of each doc; filters to relevant ones |
| `web_search` | DuckDuckGo fallback when corpus docs are irrelevant |
| `generate_answer` | granite4.1:8b generates from filtered documents |
| `grade_hallucination` | LLM checks whether answer is grounded in context |
| `regenerate` | Retry generation with an anti-hallucination prompt injection |

In [ ]:
# ── Node 1: retrieve ──────────────────────────────────────────────────────────
# Fetches candidate documents from the hybrid retriever.
# Does NOT filter — the grade_documents node decides what to keep.

def retrieve(state: GraphState) -> dict:
    """Hybrid retrieval: fetch top-10 candidates from FAISS + BM25."""
    question = state["question"]
    logger.info(f"[retrieve] query: {question[:60]}")

    # Retrieve 20 candidates from hybrid, rerank to 10
    candidates = hybrid_retriever.retrieve(question, k=20)
    documents = reranker.rerank(question, candidates, top_k=10)

    trace_entry = {"node": "retrieve", "n_docs": len(documents)}
    return {
        "documents": documents,
        "execution_trace": state.get("execution_trace", []) + [trace_entry],
    }

print("Node 'retrieve' defined.")

In [ ]:
# ── Node 2: grade_documents ───────────────────────────────────────────────────
# Uses the LLM as a judge to score each document for relevance.
# Sets retrieval_grade to:
#   "relevant"   → at least 2 documents are relevant
#   "ambiguous"  → 1 document is relevant
#   "irrelevant" → no documents are relevant → web_search will be triggered
#
# IMPORTANT: Include both title AND text in the grading prompt.
# The title often contains the most discriminative signal — without it the LLM
# may grade a relevant paper as irrelevant based on a partial abstract snippet.

GRADING_PROMPT = """You are a relevance judge.
Given a question and a document passage, determine if the document is relevant to answering the question.

Question: {question}
Document title: {title}
Document text: {text}

Is this document relevant? Respond with JSON only:
{{"relevant": true or false, "reason": "one sentence"}}"""


def grade_documents(state: GraphState) -> dict:
    """Grade each retrieved document for relevance using the LLM as judge."""
    question  = state["question"]
    documents = state["documents"]
    logger.info(f"[grade_documents] grading {len(documents)} docs...")

    relevant_docs = []
    for doc in documents:
        prompt = GRADING_PROMPT.format(
            question=question,
            title=doc.get("title", "Unknown title"),
            text=doc["text"][:400],  # truncate text; title provides the key signal
        )
        try:
            response = ollama.chat(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                format="json",
                options={"temperature": 0},
            )
            result = json.loads(response["message"]["content"])
            if result.get("relevant", False):
                relevant_docs.append(doc)
        except Exception as e:
            logger.warning(f"Grading failed for doc: {e}")

    n_relevant = len(relevant_docs)
    if n_relevant >= 2:
        grade = "relevant"
    elif n_relevant == 1:
        grade = "ambiguous"  # trigger web search to supplement
    else:
        grade = "irrelevant"  # trigger web search to replace

    logger.info(f"[grade_documents] {n_relevant}/{len(documents)} relevant → grade: {grade}")
    trace_entry = {"node": "grade_documents", "n_relevant": n_relevant, "grade": grade}

    return {
        "filtered_documents": relevant_docs,
        "retrieval_grade": grade,
        "execution_trace": state.get("execution_trace", []) + [trace_entry],
    }

print("Node 'grade_documents' defined.")

In [ ]:
# ── Node 3: web_search ────────────────────────────────────────────────────────
# Triggered when corpus documents are irrelevant or ambiguous.
# Uses DuckDuckGo to fetch fresh web results and converts them to document format
# so they can flow into the generate_answer node identically to corpus chunks.

def web_search(state: GraphState) -> dict:
    """Fallback: search the web via DuckDuckGo when corpus docs are insufficient."""
    question = state["question"]
    existing_docs = state.get("filtered_documents", [])
    logger.info(f"[web_search] searching web for: {question[:60]}")

    web_docs = []
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(question, max_results=3))
        for r in results:
            # Convert web result to same schema as corpus chunks
            web_docs.append({
                "chunk_id": f"web_{r.get('href', '')[-20:]}",
                "paper_id": r.get("href", "web"),
                "title": r.get("title", "Web result"),
                "text": r.get("body", ""),
                "source": "web",
                "score": 0.0,
            })
    except Exception as e:
        logger.warning(f"Web search failed: {e}")

    # Combine existing relevant corpus docs with web results
    combined = existing_docs + web_docs
    logger.info(f"[web_search] added {len(web_docs)} web docs → total: {len(combined)}")

    trace_entry = {"node": "web_search", "n_web_docs": len(web_docs)}
    return {
        "filtered_documents": combined,
        "execution_trace": state.get("execution_trace", []) + [trace_entry],
    }

print("Node 'web_search' defined.")

In [ ]:
# ── Node 4: generate_answer ───────────────────────────────────────────────────
# Calls granite4.1:8b with the filtered (graded) documents as context.
# granite4.1 is purpose-built for RAG — it handles structured context well
# and reliably refuses to hallucinate beyond the provided context.

GENERATION_PROMPT = """You are a research assistant specialising in machine learning and AI.
Answer the question using ONLY the information from the context documents provided.
If the context does not contain the answer, say so clearly — do not guess.
Cite the source document titles where relevant.

Context:
{context}

Question: {question}

Answer:"""


def generate_answer(state: GraphState) -> dict:
    """Generate a grounded answer using the filtered documents as context."""
    question = state["question"]
    documents = state.get("filtered_documents", state.get("documents", []))
    attempts = state.get("generation_attempts", 0)
    logger.info(f"[generate_answer] attempt {attempts+1} with {len(documents)} docs")

    context_parts = []
    for i, doc in enumerate(documents[:5], 1):  # cap at 5 docs to control prompt length
        source = doc.get("source", "arxiv")
        context_parts.append(f"[Doc {i} | {doc['title'][:50]} | {source}]\n{doc['text']}")
    context = "\n\n".join(context_parts)

    prompt = GENERATION_PROMPT.format(context=context, question=question)
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    generation = response["message"]["content"].strip()

    trace_entry = {"node": "generate_answer", "attempt": attempts + 1, "answer_len": len(generation)}
    return {
        "generation": generation,
        "generation_attempts": attempts + 1,
        "execution_trace": state.get("execution_trace", []) + [trace_entry],
    }

print("Node 'generate_answer' defined.")

In [ ]:
# ── Node 5: grade_hallucination ───────────────────────────────────────────────
# Checks whether the generated answer is grounded in the context documents.
# This is the "Corrective" step in CRAG — we don't just generate and ship,
# we verify the generation before returning it to the user.

HALLUCINATION_PROMPT = """You are a strict factual auditor.
Determine if the answer below is FULLY supported by the provided context documents.
An answer is NOT faithful if it makes any claim not found in the context.

Context:
{context}

Answer: {answer}

Respond with JSON only:
{{"faithful": true or false, "unsupported_claims": ["claim1", "claim2"] or []}}"""


def grade_hallucination(state: GraphState) -> dict:
    """Verify the generated answer is grounded in the retrieved context."""
    generation = state.get("generation", "")
    documents = state.get("filtered_documents", [])
    logger.info("[grade_hallucination] checking faithfulness...")

    context = "\n\n".join(d["text"][:400] for d in documents[:5])
    prompt = HALLUCINATION_PROMPT.format(context=context, answer=generation[:1000])

    try:
        response = ollama.chat(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            format="json",
        )
        result = json.loads(response["message"]["content"])
        faithful = result.get("faithful", True)
        unsupported = result.get("unsupported_claims", [])
    except Exception as e:
        logger.warning(f"Hallucination grading failed: {e}. Assuming faithful.")
        faithful = True
        unsupported = []

    grade = "faithful" if faithful else "hallucinated"
    logger.info(f"[grade_hallucination] grade: {grade}  unsupported claims: {len(unsupported)}")

    trace_entry = {"node": "grade_hallucination", "grade": grade, "unsupported": unsupported}
    return {
        "faithfulness_grade": grade,
        "execution_trace": state.get("execution_trace", []) + [trace_entry],
    }

print("Node 'grade_hallucination' defined.")

---

## Step 3 — Define the conditional routing functions

**Conditional edges** are the key feature that makes LangGraph more than a DAG. Instead of a fixed `A → B` edge, we define a function that reads the current state and *decides* which node to go to next.

```python
def decide_after_grading(state) -> str:
    if state["retrieval_grade"] == "relevant":
        return "generate_answer"   # good docs — go straight to generation
    else:
        return "web_search"        # bad docs — search the web first
```

The return value is a **string** that must match one of the keys in the edge mapping.

In [ ]:
# ── Routing function 1: after grade_documents ─────────────────────────────────
# Decides: go to generate_answer (docs are good) or web_search (docs are bad)

def route_after_grading(state: GraphState) -> Literal["generate_answer", "web_search"]:
    """
    Route based on retrieval quality:
      "relevant"   → documents are good, generate directly
      "ambiguous"  → supplement with web search before generating
      "irrelevant" → replace with web search results entirely
    """
    grade = state.get("retrieval_grade", "relevant")
    if grade == "relevant":
        logger.info("[route] documents relevant → generate")
        return "generate_answer"
    else:
        logger.info(f"[route] documents {grade} → web_search")
        return "web_search"


# ── Routing function 2: after grade_hallucination ─────────────────────────────
# Decides: return the answer (it's faithful) or regenerate (it hallucinated)

MAX_REGENERATIONS = 2  # prevents infinite retry loops

def route_after_hallucination_check(
    state: GraphState,
) -> Literal["__end__", "generate_answer"]:
    """
    Route based on faithfulness grade:
      "faithful"     → return answer to user
      "hallucinated" → retry generation (up to MAX_REGENERATIONS times)
    """
    grade = state.get("faithfulness_grade", "faithful")
    attempts = state.get("generation_attempts", 0)

    if grade == "faithful":
        logger.info("[route] answer faithful → END")
        return END
    elif attempts >= MAX_REGENERATIONS:
        logger.warning(f"[route] hallucination detected but max attempts reached → END")
        return END
    else:
        logger.info(f"[route] hallucination detected → regenerate (attempt {attempts+1})")
        return "generate_answer"


print("Routing functions defined.")

---

## Step 4 — Build and compile the graph

Now we wire everything together: add nodes, add edges (both fixed and conditional), set the entry point, and compile.

```
                    ┌─────────────────────────────────┐
                    │         CRAG State Machine       │
                    └─────────────────────────────────┘

START
  │
  ▼
[retrieve]              ← hybrid FAISS + BM25, reranked top-10
  │
  ▼
[grade_documents]       ← LLM grades each doc for relevance
  │
  ├── "relevant" ──────────────────────────────────────────────────┐
  │                                                                 │
  └── "ambiguous" / "irrelevant" ──► [web_search]                  │
                                          │                         │
                                          └────────────────────────┤
                                                                    ▼
                                                           [generate_answer]
                                                                    │
                                                                    ▼
                                                        [grade_hallucination]
                                                                    │
                                             ┌── "faithful" ──► END │
                                             │                      │
                                             └── "hallucinated" ◄───┘
                                                      │
                                          (max 2 retries, then END)
```

In [ ]:
# Build the LangGraph state machine
graph_builder = StateGraph(GraphState)

# ── Add nodes ─────────────────────────────────────────────────────────────────
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("grade_documents", grade_documents)
graph_builder.add_node("web_search", web_search)
graph_builder.add_node("generate_answer", generate_answer)
graph_builder.add_node("grade_hallucination", grade_hallucination)

# ── Set entry point ───────────────────────────────────────────────────────────
# The graph always starts at 'retrieve'
graph_builder.set_entry_point("retrieve")

# ── Fixed edges (always go from A to B) ──────────────────────────────────────
graph_builder.add_edge("retrieve", "grade_documents")
graph_builder.add_edge("web_search", "generate_answer")
graph_builder.add_edge("generate_answer", "grade_hallucination")

# ── Conditional edges (destination depends on state) ─────────────────────────
graph_builder.add_conditional_edges(
    "grade_documents",           # source node
    route_after_grading,         # function that returns next node name
    {
        "generate_answer": "generate_answer",   # if function returns this string
        "web_search": "web_search",             # if function returns this string
    },
)

graph_builder.add_conditional_edges(
    "grade_hallucination",
    route_after_hallucination_check,
    {
        END: END,                              # answer is faithful → stop
        "generate_answer": "generate_answer", # hallucinated → retry
    },
)

# ── Compile ───────────────────────────────────────────────────────────────────
# Compiling validates the graph (checks all nodes are reachable, etc.)
# and returns a runnable object
rag_agent = graph_builder.compile()

print("Graph compiled successfully.")
print(f"Nodes: {list(rag_agent.graph.nodes.keys())}")

---

## Step 5 — Run the agent

In [ ]:
def run_agent(question: str, verbose: bool = True) -> dict:
    """Run the CRAG agent on a single question and return the final state."""
    initial_state: GraphState = {
        "question": question,
        "documents": [],
        "filtered_documents": [],
        "retrieval_grade": "",
        "generation": "",
        "faithfulness_grade": "",
        "generation_attempts": 0,
        "execution_trace": [],
    }

    start = time.time()
    final_state = rag_agent.invoke(initial_state)
    elapsed = time.time() - start

    if verbose:
        print(f"\n{'='*70}")
        print(f"Q: {question}")
        print(f"{'='*70}")
        print(f"\nExecution path:")
        for step in final_state["execution_trace"]:
            node = step["node"]
            details = {k: v for k, v in step.items() if k != "node"}
            print(f"  → {node:25s}  {details}")
        print(f"\nRetrieval grade  : {final_state['retrieval_grade']}")
        print(f"Faithfulness     : {final_state['faithfulness_grade']}")
        print(f"Attempts         : {final_state['generation_attempts']}")
        print(f"Elapsed          : {elapsed:.1f}s")
        print(f"\nAnswer:\n{final_state['generation']}")
        print()

    return final_state

In [ ]:
# Test case 1: query well covered by the ArXiv corpus (cs.CL/cs.LG papers)
result1 = run_agent("What is functional heterogeneity in transformer attention heads?")

In [ ]:
# Test case 2: query about a very recent topic — may trigger web search fallback
result2 = run_agent("What are the latest improvements in multimodal language models in 2026?")

In [ ]:
# Test case 3: multi-concept query — tests how the agent fuses information
result3 = run_agent(
    "How does online dynamic batching improve throughput and latency for LLM inference?"
)

---

## Step 6 — Analyse execution paths

One of the main advantages of using LangGraph over a plain Python pipeline is that every execution is fully traceable. We can see exactly which nodes ran, what decisions were made, and how the system behaved for different query types.

In [ ]:
# Run the full eval set through the agent and collect execution statistics
# Using the same queries and corpus as notebooks 01 and 02 for fair comparison

papers = load_arxiv_papers(n_samples=300)

def find_relevant_ids_by_keyword(keyword, papers, top_n=3):
    kw = keyword.lower()
    return [p["id"] for p in papers if kw in p["title"].lower() or kw in p["abstract"].lower()][:top_n]

eval_queries = [
    {"question": "What is attention head heterogeneity in transformers?",
     "relevant_ids": find_relevant_ids_by_keyword("attention head", papers)},
    {"question": "How does contrastive learning work?",
     "relevant_ids": find_relevant_ids_by_keyword("contrastive learning", papers)},
    {"question": "What is dynamic batching for LLM inference?",
     "relevant_ids": find_relevant_ids_by_keyword("dynamic batching", papers)},
    {"question": "How do diffusion models generate images?",
     "relevant_ids": find_relevant_ids_by_keyword("diffusion model", papers)},
    {"question": "What are calibration methods for neural networks?",
     "relevant_ids": find_relevant_ids_by_keyword("calibration", papers)},
]

agent_results = []
for q in eval_queries:
    final_state = run_agent(q["question"], verbose=False)
    agent_results.append({
        "question": q["question"],
        "retrieval_grade": final_state["retrieval_grade"],
        "faithfulness_grade": final_state["faithfulness_grade"],
        "generation_attempts": final_state["generation_attempts"],
        "web_search_triggered": any(t["node"] == "web_search" for t in final_state["execution_trace"]),
        "answer": final_state["generation"],
        "relevant_ids": q["relevant_ids"],
        "retrieved_ids": [d.get("paper_id", "") for d in final_state.get("filtered_documents", [])],
    })

n_web = sum(r["web_search_triggered"] for r in agent_results)
n_faithful = sum(r["faithfulness_grade"] == "faithful" for r in agent_results)
n_relevant = sum(r["retrieval_grade"] == "relevant" for r in agent_results)

print("Agent execution statistics:")
print(f"  Total queries         : {len(agent_results)}")
print(f"  Corpus docs relevant  : {n_relevant}/{len(agent_results)}")
print(f"  Web search triggered  : {n_web}/{len(agent_results)} ({100*n_web/len(agent_results):.0f}%)")
print(f"  Faithful answers      : {n_faithful}/{len(agent_results)}")

In [ ]:
# Final 3-way comparison table
import pandas as pd

# Load results saved in notebooks 01 and 02
baseline = EvalResults.load(EVAL_DIR / "01_naive_rag.json")
advanced = EvalResults.load(EVAL_DIR / "02_advanced_rag.json")

# Compute agent recall@5
from src.evaluator import recall_at_k, mean_reciprocal_rank
agent_recalls, agent_mrrs = [], []
for r in agent_results:
    agent_recalls.append(recall_at_k(r["retrieved_ids"], r["relevant_ids"], k=5))
    agent_mrrs.append(mean_reciprocal_rank(r["retrieved_ids"], r["relevant_ids"]))

comparison = pd.DataFrame({
    "System": ["Naive RAG (Dense)", "Advanced RAG (Hybrid+Rerank)", "Agentic RAG (CRAG)"],
    "Recall@5":     [baseline.retrieval_metrics.get("recall@5", "—"),
                     advanced.retrieval_metrics.get("recall@5", "—"),
                     round(sum(agent_recalls)/len(agent_recalls), 4)],
    "MRR":          [baseline.retrieval_metrics.get("mrr", "—"),
                     advanced.retrieval_metrics.get("mrr", "—"),
                     round(sum(agent_mrrs)/len(agent_mrrs), 4)],
    "Web Search %": ["0%", "0%", f"{100*n_web/len(agent_results):.0f}%"],
    "Faithfulness %": ["not measured", "not measured", f"{100*n_faithful/len(agent_results):.0f}%"],
})

print("\nFinal System Comparison")
print("=" * 80)
print(comparison.to_string(index=False))

# Save traces
with open(TRACE_DIR / "eval_traces.json", "w") as f:
    json.dump(agent_results, f, indent=2, default=str)
print(f"\nTraces saved to {TRACE_DIR / 'eval_traces.json'}")

---

## Lessons Learned from Agentic RAG

### What the agent adds over the pipeline

**1. Self-correction is real and measurable.**  
The hallucination grader catches cases where the LLM overreached beyond the context. On queries where web search supplemented the corpus, faithfulness improved because the LLM had more grounded material to work with.

**2. The routing logic is the most brittle part.**  
LLM-as-judge grading is non-deterministic — the same document can receive different relevance scores across runs. Using temperature=0 (granite4.1:8b's default) helps, but the grade can still flip on borderline cases. For production, add a confidence threshold or use a dedicated smaller classifier.

**3. Web search fallback has latency costs.**  
Queries that trigger web search take 2–4× longer. The right pattern is to make web search optional (off by default in latency-sensitive settings) rather than always-on.

**4. LangGraph traces are invaluable for debugging.**  
When an answer is wrong, the trace immediately shows which node introduced the problem — was it retrieval (wrong docs), grading (incorrectly filtered a relevant doc), or generation (hallucinated beyond context)? Without traces, debugging a multi-step pipeline is guesswork.

**5. granite4.1:8b is excellent for structured JSON grading.**  
The model reliably returns valid JSON when asked — far fewer parse errors than general chat models. This is why model choice matters for the grading nodes specifically.

**6. The MAX_REGENERATIONS guard is essential.**  
Without it, a query where the corpus genuinely doesn't contain the answer creates an infinite loop: generate → hallucination detected → regenerate → hallucination detected → … The model cannot produce a faithful answer if faithful context doesn't exist.